# Ochlazování tělesa  

Uvažujme těleso, které předává teplo do svého okolí, čímž dochází k jeho ochlazování.
Předpokládejme, že v krátkých časových intervalech měříme teplotu tělesa, přičemž
jednotlivá měření jsou zatížena náhodnou chybou. Pokusíme se pro každý okamžik měření odhadnout na základě 
všech dosud naměřených hodnot aktuální teplotu tělesa a případně předpovědět teplotu v určitém
budoucím okamžiku. 

## Výchozí model
V idealizovném případě je ochalzování tělesa popsáno tzv. Newtonovým zákonem ochlazování.
Teplotu tělesa jakožto funkci času označme $x(t)$. 
Podle Newtonova zákona ochlazování odpovídá teplota tělesa v čase $t$ řešení diferenciální 
rovnice prvního řádu
$$x'(t)=-q(x(t)-T_{ok})$$
s počáteční podmínkou 
$$x(0)=T_0,$$
kde
<ul>
    <li>$q>0$ je známý konstantní koeficient přestupu tepla,</li>
    <li>$T_{ok}$ je známá konstantní teplota okolí,</li>
    <li>$T_0$ je známá teplota v čase $t=0$.</li>
</ul> 

## Zobecnění modelu
Výše popsaná počáteční úloha vychází z předpokladů, jejichž splnění lze ve skutečnosti těžko očekávat.
Pokusme se proto nyní upravit model tak, aby lépe popisoval reálnou situaci.

<ul>
    <li>Na teplotu tělesa působí další vlivy, které budeme reprezentovat jako malé náhodné výchylky.</li>
    <li>Hodnota koeficientu $q$ závisí na mnoha faktorech, jako např. velikost povrchu tělesa, 
        materiál povrchu, proudění vzduchu v okolí tělesa apod. V reálné úloze je prakticky 
        nemožné tento koeficient s dostatečnou přesností určit teoreticky. Často lze ale na základě 
        zkušenosti s podonou situací koeficient alespoň přibližně odhadnout. </li>  
    <li>Hodnota koeficientu $q$ se může v čase měnit v závislosti na změnách okolních podmínek. Protože tyto změny nejsme schopni modelovat, 
        zahrneme je do modelu rovněž jako malé náhodné výchylky.</li> 
    <li>Počáteční teplota $T_{ok}$ před začátkem měření není známá.</li>
    <li>Teplota okolí $T_{ok}$ je známa.</li>
</ul>    

## Úloha
Zobecněný model použijeme pro řešení následující úlohy.
<ul>
    <li> V časech $0<t_1<t_2<\ldots <t_n$ měříme teplotu tělesa. Měření v čase $t_k$ označme jako $y_k$</li>
    <li> Měření jsou zatížena nezávislými chybami s nulovou střední hodnotou a známým rozptylem $\sigma^2.$</li>
    <li> Na zákaldě měření $y_1, y_2, \ldots, y_k$ (přibližně) odhadněme teplotu tělesa $x_k:=x(t_k)$, jakožto podmíněněnou
        hustotu pravděpodobnosti.</li>


In [20]:
import numpy as np
from matplotlib import pyplot as plt
%matplotlib qt

In [21]:
# nacteni dat ze souboru
data = np.load("/home/janek/vsb/vyuka/odr/mereni3.npy")

In [22]:
#
data = data[:, 50:-100]

ts = data[0, 1:]
ts = ts - data[0, 0]
ys = data[1, 1:]


In [25]:
class EKF:
    def __init__(self, mu0, sig0, to):
        self.trace_t = np.array([0])
        self.trace_mu = np.array([mu0,])
        self.trace_sig = np.array([sig0,])
        self.to = to
        self.k = 0
    
    def step(self, t, y):
        self.k += 1
        
        mu_ = self.trace_mu[-1]   # \mu_{k-1}
        sig_ = self.trace_sig[-1] # \Sigma_{k-1}
        t_ = self.trace_t[-1]     # t_{k-1}
        D = t-t_                  # t_k-t_{k-1}
        
        mup = np.array([mu_[0]*(1-mu_[1]*D)+mu_[1]*self.to*D, mu_[1]])
        F = np.array([[1-mu_[1]*D, (self.to-mu_[0])*D],
                      [0, 1]])
        #Q = np.array([[1e-4, 0], 
        #              [0, 1e-10]])
        Q = np.array([[1e-4, 0], 
                      [0, 1e-10]])

        sigp = F@sig_@F.T+Q        
        G = np.array([[1, 0]])
        R = np.array([0.01])
        K = sigp@G.T@np.linalg.inv((G@sigp@G.T+R).reshape((1,1)))
        mu = mup+(K@((y-mup[0]).reshape((1,1)))).reshape(-1)
        sig = sigp-K@G@sigp
        
        #ipdb.set_trace()
        
        #self.trace_mu = np.vstack([self.trace_mu, mu])
        #self.trace_sig = np.vstack([self.trace_sig, sig])
        self.trace_mu = np.append(self.trace_mu, [mu], axis=0)
        self.trace_sig = np.append(self.trace_sig, [sig], axis=0)
        self.trace_t =  np.append(self.trace_t, t)    
        #print(K)
        #print(mu_[1], mup[1], mu[1], y)
        #print(sig)
        
        

In [35]:
#mu0 = np.array([data[1, 0], 0.1])
mu0 = np.array([50, 0.1])
sig0 = np.diag([20, 0.1])
ekf = EKF(mu0, sig0, 22.7)


In [36]:
N = 2300 # pocet pouzitych mereni
for k in range(N):
    ekf.step(ts[k], ys[k])


In [56]:
plt.figure()
tmus = ekf.trace_mu[:, 0]
tsigs = ekf.trace_sig[:, 0, 0]**0.5
ts_ext = np.concatenate([[-ts[1]], ts])[0:(N+1)]

plt.plot(ts_ext,tmus, 'b', label='střední hodnota')
plt.plot(ts_ext, tmus-2*tsigs, 'b--', alpha=0.5, label='střední hodnota $\pm 2\sigma$')
plt.plot(ts_ext, tmus+2*tsigs, 'b--', alpha=0.5)

plt.plot(ts[0:N], ys[0:N], 'r', alpha=0.5, label='měření')
plt.title('Odhad aktuální teploty pomocí Kálmánova filtru')
plt.xlabel("čas [min]")
plt.ylabel("teplota [$^\circ$C]")
plt.legend();

In [79]:
plt.figure()
cmus = ekf.trace_mu[:, 1]
csigs = ekf.trace_sig[:, 1, 1]**0.5
plt.plot(ts_ext, cmus, 'b', label='střední hodnota')
plt.plot(ts_ext, cmus-2*csigs, 'b--', alpha=0.5, label='střední hodnota $\pm 2\sigma$')
plt.plot(ts_ext, cmus+2*csigs, 'b--', alpha=0.5)
plt.title('Odhad koeficientu ochlazování')
plt.xlabel("čas [min]")
plt.ylabel("koeficient ochlazování [$^\circ C\cdot s^{-1}$]")
plt.ylim(0, 0.05)
plt.legend();
